## CECS 381 - Programming Assignment 2

**Name:** Carlos Aguilera

**Student ID:** 032455616

**Due:** August 13, 2026


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

# use fixed seed for reproducibility
random.seed(42)
np.random.seed(42)

## Problem 1: Use random sampling to estimate a constant and an integral, and study the error rate.

### 1(a) - Estimate pi with the dart method

Sample points in the unit square and count the fraction with x^2 + y^2 <= 1. Report estimates for n = 10^2, 10^4, 10^6.

In [ ]:
def estimate_pi(n):
    """
    Throw n random darts at the unit square and return an estimate of pi.

    The quarter circle takes up pi/4 of the square, so the fraction that
    lands inside times 4 gives us pi.
    """
    x = np.random.random(n)
    y = np.random.random(n)

    inside = np.sum(x**2 + y**2 <= 1)

    return 4 * inside / n

In [ ]:
for n in [10**2, 10**4, 10**6]:
    print(n, estimate_pi(n))

print()
print("true pi:", np.pi)

### 1(b) - Estimate the integral of e^(-x^2) from 0 to 1

Compare with the true value of about 0.7468.

In [ ]:
def estimate_integral(n):
    """
    Estimate the integral of e^(-x^2) from 0 to 1 using n random samples.

    The interval has width 1, so the average of the function at random
    points in [0, 1] is the integral.
    """
    x = np.random.random(n)

    return np.mean(np.exp(-x**2))

In [ ]:
estimate = estimate_integral(10**6)

print("estimate:", estimate)
print("true value:", 0.7468)
print("error:", abs(estimate - 0.7468))

### 1(c) - Error versus n on log-log axes

In [ ]:
ns = [10**k for k in range(1, 7)]
errors = [abs(estimate_pi(n) - np.pi) for n in ns]
reference = [1 / np.sqrt(n) for n in ns]

plt.loglog(ns, errors, marker='o', label='|estimate - pi|')
plt.loglog(ns, reference, label='1 / sqrt(n)')
plt.xlabel('n samples')
plt.ylabel('absolute error')
plt.legend()
plt.show()

### Comment on the observed convergence rate

On the log-log plot the error line goes down at roughly the same slope as the 1/sqrt(n) reference line, which is a slope of about -1/2. So the error shrinks like 1/sqrt(n), which is what Monte Carlo is supposed to do.

The practical downside is that this is pretty slow. To make the error 10 times smaller you need 100 times more samples. That is why the estimate at n = 10^2 can be off in the first decimal place while n = 10^6 is usually only good to about 3 decimal places.

The error line is also bumpy instead of perfectly straight, since each estimate is random and can happen to land closer or further from pi than average.

## Problem 2: Markov chain simulation and the stationary distribution

The server chain has states Up (0) and Down (1) with transition matrix P = [[0.8, 0.2], [0.6, 0.4]].

### 2(a) - Simulate the chain

Simulate T = 10^6 steps starting from Up, recording the fraction of time spent in each state.

In [ ]:
P = np.array([[0.8, 0.2],
              [0.6, 0.4]])


def simulate_chain(P, T):
    """
    Walk the chain for T steps starting from Up (state 0) and return
    the fraction of time spent in each state.
    """
    state = 0
    counts = [0, 0]

    for _ in range(T):
        counts[state] += 1

        # row `state` of P gives the chances of where we go next
        if random.random() < P[state][0]:
            state = 0
        else:
            state = 1

    return [c / T for c in counts]

In [ ]:
fractions = simulate_chain(P, 10**6)

print("fraction of time Up:", fractions[0])
print("fraction of time Down:", fractions[1])

### 2(b) - Stationary distribution by power iteration

Repeat pi <- pi P until it stops changing, then confirm it matches the empirical fractions.

In [ ]:
def power_iteration(P, steps=1000):
    """
    Start from a uniform guess and keep multiplying by P until the
    vector stops changing.
    """
    pi = np.array([0.5, 0.5])

    for _ in range(steps):
        new_pi = pi @ P
        change = np.max(np.abs(new_pi - pi))
        pi = new_pi

        if change < 1e-12:
            break

    return pi

In [ ]:
pi_power = power_iteration(P)

print("power iteration:", pi_power)
print("simulation:     ", fractions)

### 2(c) - Stationary distribution as a left eigenvector

Compute it a second way and verify all three agree.

In [ ]:
def stationary_eigenvector(P):
    """
    The stationary distribution is the left eigenvector of P with
    eigenvalue 1, which is the same as the right eigenvector of P transposed.
    """
    values, vectors = np.linalg.eig(P.T)

    # pick the eigenvector whose eigenvalue is closest to 1
    i = np.argmin(np.abs(values - 1))
    vec = np.real(vectors[:, i])

    # scale it so the probabilities add up to 1
    return vec / np.sum(vec)

In [ ]:
pi_eigen = stationary_eigenvector(P)

print("simulation:     ", fractions)
print("power iteration:", pi_power)
print("eigenvector:    ", pi_eigen)

print()
print("all three agree:", np.allclose(pi_power, pi_eigen) and np.allclose(pi_power, fractions, atol=1e-3))

## Problem 3: PageRank by power iteration

The 4-page web from Class 20: A links to B and D; B links to C; C links to A and B; D links to C.

### 3(a) - Build the link matrix H and the Google matrix G

G = alpha * H + (1 - alpha) * (1/N) with alpha = 0.85.

In [ ]:
pages = ['A', 'B', 'C', 'D']

# H[i][j] = chance of following a link from page i to page j, so rows sum to 1
H = np.array([
    [0.0, 0.5, 0.0, 0.5],   # A links to B and D
    [0.0, 0.0, 1.0, 0.0],   # B links to C
    [0.5, 0.5, 0.0, 0.0],   # C links to A and B
    [0.0, 0.0, 1.0, 0.0],   # D links to C
])

alpha = 0.85
N = len(pages)

G = alpha * H + (1 - alpha) * np.ones((N, N)) / N

print("rows of H sum to:", H.sum(axis=1))
print()
print(G)

### 3(b) - Power iteration from the uniform vector

In [ ]:
def pagerank(G, steps=1000):
    """
    Start every page with an equal share and keep multiplying by G
    until the ranks stop changing.
    """
    rank = np.ones(len(G)) / len(G)

    for _ in range(steps):
        new_rank = rank @ G
        change = np.max(np.abs(new_rank - rank))
        rank = new_rank

        if change < 1e-12:
            break

    return rank

In [ ]:
ranks = pagerank(G)

for page, r in zip(pages, ranks):
    print(page, r)

print()
print("sums to:", ranks.sum())

### 3(c) - Report the ranking

In [ ]:
order = np.argsort(ranks)[::-1]

for place, i in enumerate(order, start=1):
    print(place, pages[i], ranks[i])

### Which page wins, and why? Does link quality or quantity explain the result?

**C wins**, and the final order is C, then B, then A, then D.

Quantity by itself does not explain it. If you just count incoming links, C and B are tied: C gets links from B and D, and B gets links from A and C. So counting links would call it a tie, but C ends up clearly ahead.

What actually explains it is the **quality** of the links, which comes down to two things:

1. **The links pointing at C are undivided.** B links only to C, and D links only to C, so both of them hand over their entire rank to C. The links pointing at B are shared: A splits its rank between B and D, and C splits its rank between A and B, so B only gets half of each.
2. **C is fed by a page that is itself well ranked.** B has the second highest rank, and all of B's rank flows straight into C. Getting one undivided link from a strong page is worth more than getting several split links from weaker ones.

D is last, which makes sense: its only incoming link is half of A's rank, and A is already near the bottom.

There is also a loop here that reinforces C: C sends rank to B, and B sends all of it back to C. That cycle keeps pushing rank between the two of them, and because B passes everything to C while C only passes half back, C keeps the bigger share.